[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week11_model_governance_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 11 -- Model Governance (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Production model card, NDPA compliance, CBN AI governance, model lineage, audit trail

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Write a production model card for the retrained CreditDefault-NG model
- Write an NDPA compliance statement covering purpose, minimisation, rights, and safeguards
- Complete the CBN AI governance checklist: explainability, auditability, fairness, oversight
- Build a model lineage function queryable by decision date
- Test the governance package: given a customer ID and decision date, retrieve the full record



---

## MONDAY -- "The Production Model Card"


A model card documents what a model does, how it performs across subgroups, who it may fail for, and when it should not be used. The production model card adds: deployment date, retraining schedule, SLA thresholds, and the human oversight mechanism. Write the limitations section first -- a model card without one is marketing.

Pause and Predict -- write the three most important limitations of the credit model before reading the code. Write them as warnings to the next engineer, not as justifications to a reviewer.


### Exercise 11.1 -- Production model card

Write the complete model card. Write the limitations section first. Include aggregate and per-slice performance.


In [ ]:
# Exercise 11.1: Production model card
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: The Production Model Card --
MODEL_CARD = {
    'name': 'creditdefault-ng-v3',
    'version': '3.0.0',
    'mlflow_run_id': 'cc8a3f2d',
    'training_data_dvc_hash': 'a4f2c8d...',
    'training_period': 'Q2-Q4 2026',
    'deployment_date': '2026-07-06',
    'deployed_by': 'Temi Adeyemi',
    'approved_by': 'Dr. Emeka Obi',
    'intended_use': (
        'Automated first-stage credit default assessment. '
        'APPROVE: automated. REVIEW: human officer within 5 days. '
        'DECLINE: automated, appealable within 14 days.'
    ),
    'limitations': [
        'AUC on LIMIT_BAL > NT$800K is 0.80 -- lower than standard-limit accounts (0.91).',
        'Model trained and evaluated on a specific market; generalisation to other geographies unverified.',
        'Default labels have a 30-day lag; very recent defaults are underrepresented in training.',
    ],
    'performance': {
        'aggregate_auc': 0.871,
        'slice_standard_limit': 0.912,
        'slice_high_limit': 0.803,
    },
}


### Expected Output (illustrative)

There's no `print()` here — `MODEL_CARD` is a data structure consumed by Thursday's lineage
lookup and Friday's governance package. The line worth scrutinising: `'slice_high_limit':
0.803` versus `'slice_standard_limit': 0.912` — an 11-point AUC gap by credit-limit segment
is exactly the kind of thing a model card exists to surface *before* deployment, not discover
after.


### Monday Morning Moment

*Slack -- Monday, 9:30am.*

**Temi Adeyemi:** Is the model card just documentation or does it have legal standing under NDPA?

**Dr. Emeka Obi:** Under the NDPA it is required documentation for automated decision-making. Required means legally actionable if absent or inaccurate.

**Temi Adeyemi:** What if a customer challenges a DECLINE decision six months from now?

**Dr. Emeka Obi:** Farida needs: which model made the decision, what the basis was, the model's known limitations, and who approved it. If the governance package cannot answer all four, the decision is challengeable.

**Farida Usman:** In 18 months I need to retrieve the model card for any decision made today.

**Dr. Emeka Obi:** Which is why the lineage record is stored in MLflow alongside the model artifact -- not in a separate document.




---

## TUESDAY -- "NDPA Compliance Statement"


The Nigerian Data Protection Act (2023) requires: documented processing purpose, data minimisation, right to explanation and appeal, and safeguards against discriminatory outcomes.


### Exercise 11.2 -- NDPA compliance statement

Write the full statement covering: data controller, processing purpose, legal basis, data minimisation, automated decision-making, and all data subject rights.


In [ ]:
# Exercise 11.2: NDPA compliance statement
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: NDPA Compliance Statement --
NDPA = {
    'data_controller': 'DataFlow Infrastructure Ltd',
    'processing_purpose': (
        'Assessment of credit default risk. '
        'Legal basis: legitimate interest and contractual necessity.'
    ),
    'data_minimisation': (
        '23 features used: LIMIT_BAL, PAY_0-PAY_6, BILL_AMT1-6, PAY_AMT1-6, '
        'SEX, EDUCATION, MARRIAGE, AGE. '
        'No sensitive personal data categories (ethnicity, religion) processed.'
    ),
    'automated_decisions': (
        'APPROVE: fully automated. '
        'REVIEW: human officer required within 5 business days. '
        'DECLINE: automated with right to explanation via /explain?customer_id=X '
        'and human appeal within 14 calendar days.'
    ),
    'retention': '7 years per applicable regulations.',
    'data_subject_rights': {
        'right_to_explanation': '/explain?customer_id=X (SHAP-based)',
        'right_to_human_review': 'All DECLINE decisions appealable within 14 days',
        'right_to_erasure': 'Supported; does not modify the deployed model'
    },
}


### Expected Output (illustrative)

No `print()` output — `NDPA` is a compliance record, not a script. The clause worth reading
twice: `'data_minimisation'` states explicitly that no sensitive personal data categories
(ethnicity, religion) are processed — a genuine NDPA-relevant claim needs to be true of your
actual feature set, not just asserted in a dict.



---

## WEDNESDAY -- "CBN AI Governance Checklist"


CBN AI governance guidelines require credit models to be explainable (basis communicated to customer), auditable (complete audit trail for every prediction), fair (tested for discriminatory outcomes), and human-overseen (qualified officer can override).


### Exercise 11.3 -- CBN checklist

Complete the checklist. For each of the four requirements: COMPLIANT or NON-COMPLIANT, cited evidence, and any gaps.


In [ ]:
# Exercise 11.3: CBN checklist
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: CBN AI Governance Checklist --
CBN = {
    'explainability': {
        'method': 'SHAP values per prediction', 'endpoint': '/explain?customer_id=X',
        'example': ('Declined because: (1) two-month payment delay in most recent statement; '
                    '(2) bill amount is 94% of credit limit; '
                    '(3) no payment made in previous month.'),
        'status': 'COMPLIANT'},
    'auditability': {
        'prediction_log': 'All predictions logged with customer_id, timestamp, features, probability, decision',
        'retention': '7 years', 'status': 'COMPLIANT'},
    'fairness': {
        'attributes_tested': ['SEX', 'EDUCATION', 'MARRIAGE', 'AGE'],
        'method': 'Per-group AUC and approval rate disparity',
        'finding': 'No significant disparity by SEX (chi-sq p=0.38)',
        'status': 'COMPLIANT'},
    'human_oversight': {
        'APPROVE': 'Automated', 'REVIEW': 'Human officer within 5 days',
        'DECLINE': 'Human appeal within 14 days', 'status': 'COMPLIANT'},
}


### Expected Output (illustrative)

No `print()` output — `CBN` documents four governance dimensions, each with a `'status'` of
`'COMPLIANT'`. Treat each `'COMPLIANT'` as a claim requiring evidence, not a checkbox: the
`fairness` entry's claim (`chi-sq p=0.38`, no significant disparity by SEX) should be a number
you actually computed against your holdout set, not a placeholder.



---

## THURSDAY -- "Model Lineage and Audit Trail"


The lineage function answers: which model made this prediction on this date? It must work for any historical date -- not just today's model.

Pause and Predict -- why does the lineage function filter by decision_date rather than returning the current production model? What would go wrong if a customer challenged a decision from 6 months ago and the function returned today's model?


### STOP AND TRACE

Before running:

versions = client.get_latest_versions('credit-default-v2', stages=['Production'])
target_ts = pd.Timestamp('2026-06-08').timestamp() * 1000

Why multiply by 1000 when converting the timestamp?
What does get_latest_versions return if two versions are both in 'Production'?
Why does the lineage function filter by creation_timestamp <= target_ts?
Why this matters: a decision on January 15th was made by the January model, not the July retrained model.


In [ ]:
# Exercise 11.4: STOP AND TRACE -- MLflow timestamps
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Model Lineage and Audit Trail --
from mlflow.tracking import MlflowClient
import pandas as pd

def get_model_lineage(decision_date: str) -> dict:
    """Return the governance record for a specific decision date."""
    client = MlflowClient()
    versions = client.get_latest_versions('credit-default-v2', stages=['Production'])
    target_ts = pd.Timestamp(decision_date).timestamp() * 1000
    # Find the version that was in Production on the decision date
    active = sorted(
        [v for v in versions if v.creation_timestamp <= target_ts],
        key=lambda v: v.creation_timestamp)[-1]
    run = client.get_run(active.run_id)
    return {
        'model_version':     active.version,
        'mlflow_run_id':     active.run_id,
        'training_data_hash':run.data.tags.get('dvc.data_hash', 'not set'),
        'git_commit':        run.data.tags.get('git.commit', 'not set'),
        'deployed_by':       active.tags.get('deployed_by', 'not set'),
        'approved_by':       active.tags.get('approved_by', 'not set'),
    }


### Expected Output (illustrative — depends on your MLflow registry state)

```
{'model_version': '3', 'mlflow_run_id': 'cc8a3f2d91b74e0a9f7c3e5b6a1d8f42',
 'training_data_hash': 'a4f2c8d1e9b03f5a7c6d2e8f1a4b9c7d', 'git_commit': 'e4b7a91',
 'deployed_by': 'Temi Adeyemi', 'approved_by': 'Dr. Emeka Obi'}
```
This function answers a specific auditor question: *"which exact model made the decision on
this date, and who approved it?"* — if any of these fields comes back `'not set'`, that's a
governance gap to close before the model is truly audit-ready, not just a missing print value.



---

## FRIDAY -- "The Friday Build: Governance Package"


Assemble all four governance documents. Link the package from the MLflow model registry. Test: customer_id=4823, decision date 2026-06-08 -- retrieve the full governance record and verify it shows the correct model version for that date.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Governance Package --
import json
package = {'model_card': MODEL_CARD, 'ndpa': NDPA, 'cbn': CBN,
           'lineage': get_model_lineage('2026-06-08')}
with open('outputs/governance_package_v3.json', 'w') as f:
    json.dump(package, f, indent=2)
print('Governance package saved')

# Link from MLflow
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.set_model_version_tag('credit-default-v2', '3', 'governance_package',
                             'outputs/governance_package_v3.json')

# Test
lineage = get_model_lineage('2026-06-08')
print(f"Decision on 2026-06-08 made by model v{lineage['model_version']}")
print(f"Approved by: {lineage['approved_by']}")


### Expected Output (illustrative)

```
Governance package saved
Decision on 2026-06-08 made by model v3
Approved by: Dr. Emeka Obi
```
The three files this week produces (`model_card`, `NDPA` statement, `CBN` checklist) plus the
lineage lookup together form `governance_package_v3.json` — this is the artifact an external
auditor or regulator would actually be handed, not a summary written for them after the fact.


### Friday Workplace Moment

*DataFlow -- Friday standup.*

**Dr. Emeka Obi:** Governance test. Customer 4823, DECLINE, June 8th.

**Temi Adeyemi:** Model version 2. Run ID a3f8c91. DVC hash 4d2c8a1. Deployed by Temi Adeyemi. Approved by Dr. Emeka Obi.

**Dr. Emeka Obi:** Explanation for that customer?

**Temi Adeyemi:** Top three SHAP: PAY_0=2 (two-month delay), BILL_AMT1 at 92% of credit limit, PAY_AMT1 below average.

**Farida Usman:** I can show that to the customer. That is a defensible explanation.



## YOUR WEEK 11 CHECKLIST

Check each item before moving on.

- [ ] Model card limitations section was written first, before the performance section.
- [ ] NDPA statement addresses all four requirements with specific evidence.
- [ ] CBN checklist is complete with COMPLIANT/NON-COMPLIANT and cited evidence.
- [ ] Lineage function returns the correct model version for a historical decision date.
- [ ] Governance package linked from MLflow model registry and version-controlled.


## Challenge Task

> **Core path:** Implement the /explain endpoint: given a customer_id, return top 3 SHAP features in plain English suitable for a customer-facing letter.

> **Advanced path:** Build a governance audit trail logging every model card change with author, date, and diff.


## Common Mistakes

**Writing the model card after deployment:** A model card written after deployment to justify decisions already made is a liability document.

**Model card without limitations:** A model card that only documents performance is marketing. Limitations are what legal teams read first.

**Governance not queryable by decision date:** A governance record that cannot answer 'which model made this decision on this date' is not useful in a legal challenge.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)